In [1]:
# Import necessary libraries
import json
from langchain_openai import ChatOpenAI

c:\Users\catha\Documents\agentic-protein-reconstruction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load a sample fragmented protein
with open("../data/fragmented_ecoli.jsonl") as f:
    sample = json.loads(f.readline())

fragments = sample["fragments"]
original = sample["ecoli_original"]
print(f"Original Protein: {original}")
print(f"Original Length: {len(original)}")
print(f"Fragmented Protein: {fragments}")
print(f"Number of Fragments: {len(fragments)}")

Original Protein: MSQPITRENFDEWMIPVYAPAPFIPVRGEGSRLWDQQGKEYIDFAGGIAVNALGHAHPELREALNEQASKFWHTGNGYTNESVLRLAKKLIDATFADRVFFCNSGAEANEAALKLARKFAHDRYGSHKSGIVAFKNAFHGRTLFTVSAGGQPAYSQDFAPLPPDIRHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRELCDRHNALLIFDEVQTGVGRTGELYAYMHYGVTPDLLTTAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKVLELINTPEMLNGVKQRHDWFVERLNIINHRYGLFNEVRGLGLLIGCVLNADYAGQAKQISQEAAKAGVMVLIAGGNVVRFAPALNVSEEEVTTGLDRFAVACEHFVSRGSS
Original Length: 406
Fragmented Protein: ['GSS', 'LAR', 'EYIDFAGGIAVNALGHAHPELR', 'NAFHGR', 'FWHTGNGYTNESVLR', 'K', 'HNALLIFDEVQTGVGR', 'LWDQQGK', 'QISQEAAK', 'FAHDR', 'ENFDEWMIPVYAPAPFIPVR', 'HAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLR', 'SGIVAFK', 'FAVACEHFVSR', 'VFFCNSGAEANEAALK', 'FAPALNVSEEEVTTGLDR', 'GEGSR', 'TLFTVSAGGQPAYSQDFAPLPPDIR', 'LIDATFADR', 'VLELINTPEMLNGVK', 'EALNEQASK', 'ELCDR', 'YGLFNEVR', 'YGSHK', 'AGVMVLIAGGNVVR', 'K', 'HDWFVER', 'LNIINHR', 'TGELYAYMHYGVTPDLLTTAK', 'GLGLLIGCVLNADYAGQAK', 'ALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGK', 'MSQPITR', 'QR'

In [3]:
SYSTEM_PROMPT = (
    "You are an expert protein reconstruction assistant. "
    "You are given peptide fragments from a single protein in random order. "
    "Your task is to reconstruct the original protein sequence exactly by determining the correct order of the peptides. "
    "Do not alter the amino acid sequences. "
    "Return only the reconstructed protein sequence, without any explanations or extra text."
)

user_prompt = (
    "Example 1:\n"
    "Shuffled peptides: ['ASEFGVVLSVDALK', 'LSR', 'MELK', 'QSPLGVGIGGGGGGGGGGSCGGQGGGCGGCSNGCSGGNGGSGGSGSHI']\n"
    "Original sequence: MELKASEFGVVLSVDALKLSRQSPLGVGIGGGGGGGGGGSCGGQGGGCGGCSNGCSGGNGGSGGSGSHI\n\n"
    "Now reconstruct the original protein sequence from the following shuffled peptides:\n"
    f"{fragments}\n\n"
    "Return only the reconstructed protein sequence exactly."
)

In [4]:
# Test on the sample
llm = ChatOpenAI(model="gpt-5.2", temperature=0)

response = llm.invoke([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt},
])

result = response.content
print(f"Original:      {original}")
print(f"Reconstructed: {result}")
print(f"Match:         {result == original}")

Original:      MSQPITRENFDEWMIPVYAPAPFIPVRGEGSRLWDQQGKEYIDFAGGIAVNALGHAHPELREALNEQASKFWHTGNGYTNESVLRLAKKLIDATFADRVFFCNSGAEANEAALKLARKFAHDRYGSHKSGIVAFKNAFHGRTLFTVSAGGQPAYSQDFAPLPPDIRHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRELCDRHNALLIFDEVQTGVGRTGELYAYMHYGVTPDLLTTAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKVLELINTPEMLNGVKQRHDWFVERLNIINHRYGLFNEVRGLGLLIGCVLNADYAGQAKQISQEAAKAGVMVLIAGGNVVRFAPALNVSEEEVTTGLDRFAVACEHFVSRGSS
Reconstructed: MSQPITREYIDFAGGIAVNALGHAHPELRSGIVAFKNAFHGRLNIINHRYGSHKHDWFVERFWHTGNGYTNESVLRLIDATFADRTGELYAYMHYGVTPDLLTTAKAGVMVLIAGGNVVRLWDQQGKGEGSRGSSFAVACEHFVSRGLGLLIGCVLNADYAGQAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKVFFCNSGAEANEAALKEALNEQASKQISQEAAKHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRVLELINTPEMLNGVKFAPALNVSEEEVTTGLDRELCDRYGLFNEVRENFDEWMIPVYAPAPFIPVRTLFTVSAGGQPAYSQDFAPLPPDIRLARKQRKLAK
Match:         False
